In [1]:
!pip install pandas


hs_rag_qwen
¶
Complete revised hybrid RAG notebook using BGE-M3 for retrieval and Qwen2.5-3B-Instruct for answer generation.

In [2]:
import pandas as pd
import math
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import torch
import gradio as gr
import faiss
from rank_bm25 import BM25Okapi
import re
import warnings
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from pathlib import Path
import zipfile
import json

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

warnings.filterwarnings('ignore')

# Create common project folders used throughout the notebook.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
SEARCH_REVIEW_DIR = Path("search_review")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SEARCH_REVIEW_DIR.mkdir(exist_ok=True)

# NLTK stop words are helpful for BM25 and word-overlap scoring.
# This keeps the notebook runnable even on a fresh machine.
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()


In [3]:
DEPARTMENT_NAME = "Computer Science and Electrical Engineering"
SOURCE_ROOT = Path("DepCourseInfo")
ZIP_PATH_CANDIDATES = [Path("DepCourseInfo.zip"), Path("/mnt/data/DepCourseInfo.zip")]
PDF_FOLDER_NAME = "downloaded_forms_and_worksheets"

# Prefixes that are common in CS, CE, cybersecurity, networking, data science, AI, and EE/engineering course material.
# Keep this broad because the EECS pages may contain interdisciplinary course documents too.
EECS_PREFIXES = {
    "CAP", "CDA", "CEN", "CGS", "CIS", "CNT", "COP", "COT", "CSC",
    "EEE", "EEL", "EGN", "EGS", "EIN", "EML", "IDC", "ISC", "STA", "MAP", "MAD"
}

# # Avoid carrying over biomedical/healthcare-oriented documents from the old topic.
# # A CS/EE course with a bioinformatics title is still kept if its prefix is part of EECS_PREFIXES.
# BIOMEDICAL_EXCLUDE_TERMS = {
#     "bme", "biomedical", "bioengineering", "bio-systems", "stem cell", "tissue engineering"
# }

def normalize_whitespace(text):
    return " ".join(str(text).replace("\x00", " ").split())


def clean_title_from_filename(filename):
    stem = Path(filename).stem
    stem = re.sub(r"[_-]+", " ", stem)
    stem = re.sub(r"\s+", " ", stem).strip()
    return stem.title()


def ensure_source_files_available(source_root=SOURCE_ROOT):
    """Use an existing DepCourseInfo folder, or unzip DepCourseInfo.zip if needed."""
    if source_root.exists():
        print(f"Using existing source folder: {source_root.resolve()}")
        return source_root

    zip_path = next((p for p in ZIP_PATH_CANDIDATES if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            "Could not find DepCourseInfo folder or DepCourseInfo.zip. "
            "Place the ZIP next to this notebook or update ZIP_PATH_CANDIDATES."
        )

    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(Path("."))

    if not source_root.exists():
        raise FileNotFoundError(f"Expected extracted folder was not found: {source_root}")

    print(f"Extracted source folder: {source_root.resolve()}")
    return source_root


def guess_document_type(filename, linked_url="", title=""):
    text = f"{filename} {linked_url} {title}".lower()

    if any(word in text for word in ["worksheet", "application", "audit", "form"]):
        return "program_form_or_worksheet"
    if "certificate" in text or "minor" in text or "concentration" in text:
        return "certificate_minor_or_concentration"
    if re.search(r"\b[a-z]{3}\s*[-_ ]?\d{4}", text):
        return "course_syllabus"
    if "syllab" in text:
        return "course_syllabus"
    return "department_document"


def is_relevant_eecs_document(prefix="", title="", filename="", doc_type=""):
    """Return True for CS/EE department material and False for biomedical/healthcare carryover."""
    prefix = str(prefix).upper().strip()
    combined = f"{title} {filename}".lower()

    if prefix:
        return prefix in EECS_PREFIXES

    # if any(term in combined for term in BIOMEDICAL_EXCLUDE_TERMS):
    #     return False

    # Keep program forms, worksheets, certificates, and general department files that do not have a course prefix.
    return True


def extract_pdf_text(pdf_path, max_pages=None):
    """Extract text from a PDF. Returns an empty string for scanned/unreadable PDFs."""
    if PdfReader is None:
        raise ImportError("pypdf is required. Install it with: pip install pypdf")

    try:
        reader = PdfReader(str(pdf_path))
        pages = reader.pages if max_pages is None else reader.pages[:max_pages]
        page_text = []
        for page in pages:
            try:
                page_text.append(page.extract_text() or "")
            except Exception:
                page_text.append("")
        return normalize_whitespace("\n".join(page_text))
    except Exception as exc:
        print(f"Warning: could not read PDF {pdf_path.name}: {exc}")
        return ""


def build_department_corpus(source_root=SOURCE_ROOT, data_dir=DATA_DIR):
    """
    Build a retrieval corpus for CS/EE department information.

    Output columns intentionally include the original template's required columns:
    - doc_id
    - text

    Additional metadata columns are kept for better summaries and inspection.
    """
    source_root = ensure_source_files_available(source_root)
    data_dir.mkdir(exist_ok=True)

    course_csv = source_root / "course_listings.csv"
    pdf_dir = source_root / PDF_FOLDER_NAME

    rows = []
    linked_file_lookup = {}

    if course_csv.exists():
        courses_df = pd.read_csv(course_csv)
        courses_df = courses_df.fillna("")

        # Map downloaded PDF filenames back to their source URLs when possible.
        for _, row in courses_df.iterrows():
            linked = str(row.get("linked_file_or_page", "")).strip()
            if linked.lower().endswith(".pdf"):
                linked_file_lookup[Path(linked).name.lower()] = linked

        # Add one clean course-listing document per unique course/title/source combination.
        course_docs = courses_df.copy()
        course_docs["course_key"] = (
            course_docs["course_code"].astype(str) + " | " +
            course_docs["title_or_context"].astype(str) + " | " +
            course_docs["level"].astype(str)
        )
        course_docs = course_docs.drop_duplicates("course_key")

        for i, row in course_docs.iterrows():
            prefix = str(row.get("prefix", "")).upper().strip()
            course_code = str(row.get("course_code", "")).strip()
            title = str(row.get("title_or_context", "")).strip()

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename="", doc_type="course_listing"):
                continue
            level = str(row.get("level", "")).strip()
            source_page = str(row.get("source_page", "")).strip()
            linked = str(row.get("linked_file_or_page", "")).strip()
            raw_text = str(row.get("raw_text", "")).strip()

            # Keep broad EECS results, but avoid empty/noisy rows.
            if not course_code and not title and not raw_text:
                continue

            doc_id = f"course_{len(rows)+1:04d}_{course_code.replace(' ', '_') or 'unknown'}"
            labeled_text = normalize_whitespace(f"""
Document_Type = course_listing,
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {row.get('number', '')},
Level = {level},
Source_Page = {source_page},
Linked_File_or_Page = {linked},
File_Name = ,
Content = {raw_text if raw_text else f'{course_code} {title}'}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": "course_listing",
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": str(row.get("number", "")).strip(),
                "level": level,
                "source_page": source_page,
                "linked_file_or_page": linked,
                "file_name": "",
                "text": labeled_text,
            })
    else:
        print(f"Warning: course listings file was not found: {course_csv}")

    if pdf_dir.exists():
        pdf_paths = sorted(pdf_dir.rglob("*.pdf"))
        print(f"Reading {len(pdf_paths)} PDFs from {pdf_dir} ...")

        for pdf_path in pdf_paths:
            file_name = pdf_path.name
            linked = linked_file_lookup.get(file_name.lower(), "")
            title = clean_title_from_filename(file_name)
            doc_type = guess_document_type(file_name, linked, title)
            pdf_text = extract_pdf_text(pdf_path)

            # Try to identify course code metadata from the filename or extracted text.
            course_match = re.search(r"\b([A-Z]{3})\s*[-_ ]?(\d{4})\b", f"{file_name} {pdf_text[:500]}", flags=re.IGNORECASE)
            prefix = course_match.group(1).upper() if course_match else ""
            number = course_match.group(2) if course_match else ""
            course_code = f"{prefix} {number}".strip()
            level = "graduate" if number.startswith(("5", "6")) else "undergraduate" if number.startswith(("1", "2", "3", "4")) else ""

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename=file_name, doc_type=doc_type):
                continue

            content = pdf_text if pdf_text else "No extractable PDF text was found. Use the filename and metadata for retrieval."
            doc_id = f"pdf_{len(rows)+1:04d}_{Path(file_name).stem[:60]}"
            labeled_text = normalize_whitespace(f"""
Document_Type = {doc_type},
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {number},
Level = {level},
Source_Page = ,
Linked_File_or_Page = {linked},
File_Name = {file_name},
Content = {content}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": doc_type,
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": number,
                "level": level,
                "source_page": "",
                "linked_file_or_page": linked,
                "file_name": file_name,
                "text": labeled_text,
            })
    else:
        print(f"Warning: PDF folder was not found: {pdf_dir}")

    corpus_df = pd.DataFrame(rows)
    if corpus_df.empty:
        raise ValueError("No department documents were found. Check SOURCE_ROOT and PDF_FOLDER_NAME.")

    corpus_path = data_dir / "corpus.csv"
    corpus_df.to_csv(corpus_path, index=False)
    print(f"Department corpus saved to: {corpus_path}")
    print(f"Documents created: {len(corpus_df)}")
    print(corpus_df["document_type"].value_counts())
    return corpus_df


def load_corpus(corpus_name="corpus", rebuild=False):
    corpus_fullcsvname = DATA_DIR / f"{corpus_name}.csv"

    if rebuild or not corpus_fullcsvname.exists():
        print("Building a new CS/EE department corpus...")
        corpus_df = build_department_corpus()
    else:
        corpus_df = pd.read_csv(corpus_fullcsvname).fillna("")

    print(f"Corpus: {len(corpus_df)} documents")
    print(f"Columns: {list(corpus_df.columns)}")
    print()
    print("First 3 documents:")
    for _, row in corpus_df.head(3).iterrows():
        preview = str(row["text"])[:220] + "..." if len(str(row["text"])) > 220 else str(row["text"])
        print(f"  {row['doc_id']}: {preview}")
    print()

    return corpus_df


In [4]:
corpus_name = "corpus"

# Use True the first time you switch from the old healthcare corpus to the CS/EE corpus.
# After data/corpus.csv has been created, you can set this to False for faster reruns.
REBUILD_CORPUS = True

corpus_df = load_corpus(corpus_name, rebuild=REBUILD_CORPUS)


Building a new CS/EE department corpus...
Using existing source folder: C:\Users\xande\Projects_Py\EE\eecs_search_review_update\DepCourseInfo
Reading 266 PDFs from DepCourseInfo\downloaded_forms_and_worksheets ...
Department corpus saved to: data\corpus.csv
Documents created: 532
document_type
course_listing               275
course_syllabus              173
program_form_or_worksheet     74
department_document           10
Name: count, dtype: int64
Corpus: 532 documents
Columns: ['doc_id', 'document_type', 'title', 'course_code', 'prefix', 'number', 'level', 'source_page', 'linked_file_or_page', 'file_name', 'text']

First 3 documents:
  course_0001_CAP_5615: Document_Type = course_listing, Title = Intro to Neural Networks, Course_Code = CAP 5615, Prefix = CAP, Number = 5615, Level = graduate, Source_Page = https://www.fau.edu/engineering/eecs/research/nrt/faculty/collaborato...
  course_0002_CAP_5615: Document_Type = course_listing, Title = Introduction to Neural Networks, Course_Code

Chunked Retrieval Corpus
¶
This block keeps the original document-level
corpus_df
, but creates a second retrieval corpus named
active_corpus_df
/
retrieval_df
.
BM25 and FAISS should index
active_corpus_df
, where each row is an overlapping chunk from a source document. This lets hybrid search retrieve specific parts of longer worksheets, forms, PDFs, and course lists instead of matching only document headings or filenames.

In [5]:
# ============================================================
# Build a chunked retrieval corpus for BM25 + FAISS
# ============================================================
# Why this exists:
# - corpus_df keeps one row per original source document.
# - retrieval_df / active_corpus_df keeps many smaller chunks per document.
# - BM25 and FAISS index the chunks, which improves recall for long PDFs/forms.
# - Evaluation qrels are expanded from original document IDs to chunk IDs later.

CHUNKED_CORPUS_NAME = "corpus_chunks"
REBUILD_RETRIEVAL_CHUNKS = True
CHUNK_SIZE_WORDS = 750
CHUNK_OVERLAP_WORDS = 75
MIN_CHUNK_WORDS = 35


def extract_content_from_labeled_text(text):
    """Return the Content field from the labeled corpus text when possible."""
    text = normalize_whitespace(text)
    match = re.search(r"\bContent\s*=\s*(.*)$", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        content = match.group(1).strip(" ,")
        if content:
            return content
    return text


def chunk_words_with_overlap(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS, min_words=MIN_CHUNK_WORDS):
    """Split text into overlapping word chunks."""
    text = normalize_whitespace(text)
    words = text.split()

    if not words:
        return []

    if len(words) <= chunk_size:
        return [" ".join(words)] if len(words) >= min_words or len(words) > 0 else []

    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk = " ".join(words[start:end]).strip()

        # Avoid adding tiny tail chunks unless this is the only chunk.
        if len(chunk.split()) >= min_words:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks


def looks_like_low_value_repetition(text, max_repetition_ratio=0.35):
    """Filter chunks that are mostly repeated OCR/PDF noise, such as 'phd phd phd ...'."""
    words = re.findall(r"[A-Za-z]+", str(text).lower())
    if len(words) < 40:
        return False
    counts = Counter(words)
    most_common_word, most_common_count = counts.most_common(1)[0]
    return most_common_count / max(len(words), 1) > max_repetition_ratio


def metadata_prefix_for_chunk(row, chunk_index, chunk_count):
    """Attach enough metadata for retrieval without letting metadata overwhelm the chunk text."""
    title = str(row.get("title", "")).strip()
    document_type = str(row.get("document_type", "")).strip()
    course_code = str(row.get("course_code", "")).strip()
    prefix = str(row.get("prefix", "")).strip()
    number = str(row.get("number", "")).strip()
    level = str(row.get("level", "")).strip()
    file_name = str(row.get("file_name", "")).strip()
    source_page = str(row.get("source_page", "")).strip()
    linked = str(row.get("linked_file_or_page", "")).strip()
    parent_doc_id = str(row.get("doc_id", "")).strip()

    return normalize_whitespace(f"""
Document_Type = {document_type},
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {number},
Level = {level},
Parent_Doc_ID = {parent_doc_id},
Chunk_Index = {chunk_index} of {chunk_count},
Source_Page = {source_page},
Linked_File_or_Page = {linked},
File_Name = {file_name},
Content =
""")


def build_chunked_retrieval_corpus(corpus_df, data_dir=DATA_DIR):
    """
    Create chunk-level rows for retrieval.

    Important columns:
    - doc_id: unique chunk ID used by BM25/FAISS
    - parent_doc_id: original source document ID
    - text: metadata + chunk text used for indexing
    - chunk_text: just the chunk body used for RAG context compression
    """
    rows = []

    for _, row in corpus_df.fillna("").iterrows():
        parent_doc_id = str(row.get("doc_id", "")).strip()
        full_text = str(row.get("text", ""))
        content = extract_content_from_labeled_text(full_text)
        chunks = chunk_words_with_overlap(content)

        if not chunks:
            chunks = [normalize_whitespace(full_text)]

        chunk_count = len(chunks)
        for chunk_index, chunk_text in enumerate(chunks, start=1):
            if looks_like_low_value_repetition(chunk_text):
                continue

            chunk_id = f"{parent_doc_id}::chunk_{chunk_index:03d}"
            prefix = metadata_prefix_for_chunk(row, chunk_index, chunk_count)
            index_text = normalize_whitespace(prefix + " " + chunk_text)

            rows.append({
                "doc_id": chunk_id,
                "parent_doc_id": parent_doc_id,
                "chunk_index": chunk_index,
                "chunk_count": chunk_count,
                "chunk_text": normalize_whitespace(chunk_text),
                "text": index_text,
                "document_type": str(row.get("document_type", "")).strip(),
                "title": str(row.get("title", "")).strip(),
                "course_code": str(row.get("course_code", "")).strip(),
                "prefix": str(row.get("prefix", "")).strip(),
                "number": str(row.get("number", "")).strip(),
                "level": str(row.get("level", "")).strip(),
                "source_page": str(row.get("source_page", "")).strip(),
                "linked_file_or_page": str(row.get("linked_file_or_page", "")).strip(),
                "file_name": str(row.get("file_name", "")).strip(),
            })

    retrieval_df = pd.DataFrame(rows)
    if retrieval_df.empty:
        raise ValueError("No retrieval chunks were created. Check corpus_df and chunking settings.")

    output_path = data_dir / f"{CHUNKED_CORPUS_NAME}.csv"
    retrieval_df.to_csv(output_path, index=False)

    print(f"Chunked retrieval corpus saved to: {output_path}")
    print(f"Original source documents: {len(corpus_df)}")
    print(f"Retrieval chunks created: {len(retrieval_df)}")
    print(f"Average chunks per source document: {len(retrieval_df) / max(len(corpus_df), 1):.2f}")
    print("Chunk length summary, in words:")
    print(retrieval_df["chunk_text"].astype(str).apply(lambda x: len(x.split())).describe())

    return retrieval_df


def load_or_build_retrieval_chunks(corpus_df, rebuild=REBUILD_RETRIEVAL_CHUNKS):
    chunk_path = DATA_DIR / f"{CHUNKED_CORPUS_NAME}.csv"

    if rebuild or not chunk_path.exists():
        retrieval_df = build_chunked_retrieval_corpus(corpus_df)
    else:
        retrieval_df = pd.read_csv(chunk_path).fillna("")

    print(f"Active retrieval corpus: {len(retrieval_df)} chunks")
    print(f"Columns: {list(retrieval_df.columns)}")
    print("\nFirst 3 retrieval chunks:")
    for _, row in retrieval_df.head(3).iterrows():
        preview = str(row["chunk_text"])[:220] + "..." if len(str(row["chunk_text"])) > 220 else str(row["chunk_text"])
        print(f"  {row['doc_id']}  | parent={row.get('parent_doc_id', '')}")
        print(f"    {preview}")
    print()

    return retrieval_df


retrieval_df = load_or_build_retrieval_chunks(corpus_df)

# Use active_corpus_df for retrieval, indexing, and RAG context.
# Keep corpus_df unchanged for full-document metadata and inspection.
active_corpus_df = retrieval_df
RETRIEVAL_IS_CHUNKED = "parent_doc_id" in active_corpus_df.columns


Chunked retrieval corpus saved to: data\corpus_chunks.csv
Original source documents: 532
Retrieval chunks created: 577
Average chunks per source document: 1.08
Chunk length summary, in words:
count    577.000000
mean     186.350087
std      225.877130
min        2.000000
25%        5.000000
50%      135.000000
75%      268.000000
max      750.000000
Name: chunk_text, dtype: float64
Active retrieval corpus: 577 chunks
Columns: ['doc_id', 'parent_doc_id', 'chunk_index', 'chunk_count', 'chunk_text', 'text', 'document_type', 'title', 'course_code', 'prefix', 'number', 'level', 'source_page', 'linked_file_or_page', 'file_name']

First 3 retrieval chunks:
  course_0001_CAP_5615::chunk_001  | parent=course_0001_CAP_5615
    Instructor for CAP 5615 Intro to Neural Networks
  course_0002_CAP_5615::chunk_001  | parent=course_0002_CAP_5615
    CAP 5615 Introduction to Neural Networks
  course_0003_CAP_5625::chunk_001  | parent=course_0003_CAP_5625
    CAP 5625



In [6]:
DEFAULT_DEPARTMENT_QUERIES = [('q01', 'Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.'), ('q02', 'Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.'), ('q03', 'Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.'), ('q04', 'Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.'), ('q05', 'Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.'), ('q06', 'Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering graduate program information, and electrical engineering degree requirements.'), ('q07', 'Find cybersecurity certificate and minor documents, cyber security worksheets, cybersecurity applications, and security-related courses such as cryptography, network security, and data security.'), ('q08', 'Find data science and analytics certificate documents, DSA concentration worksheets, CSDA or computer science data analytics program sheets, and related data science courses.'), ('q09', 'Find machine learning, artificial intelligence, neural networks, deep learning, computer vision, and NLP course syllabi.'), ('q10', 'Find data mining, information retrieval, web mining, big data analytics, and database-related course syllabi.'), ('q11', 'Find software engineering courses and syllabi covering software requirements engineering, software testing, software architecture, software maintenance, and object-oriented design.'), ('q12', 'Find electrical engineering course syllabi for circuits, electronics, signal processing, communications, control systems, antennas, RF, power systems, and smart grid topics.'), ('q13', 'Find computer architecture, digital logic, microprocessor, embedded systems, VLSI, and computer design course syllabi.'), ('q14', 'Find general EECS student forms and administrative documents such as audit forms, application forms, graduate pathway scholarships, low residency PhD forms, and conflict of interest disclosure forms.')]


def create_default_queries(query_path):
    query_path = Path(query_path)
    query_path.parent.mkdir(parents=True, exist_ok=True)
    query_df = pd.DataFrame(DEFAULT_DEPARTMENT_QUERIES, columns=["query_id", "query_text"])
    query_df.to_csv(query_path, index=False)
    print(f"Default CS/EE department queries saved to: {query_path}")
    return query_df


def load_premade_queries(query_path=None, rebuild=False):
    """Load the starter EECS query list from search_review/eecs_queries.csv."""
    query_path = Path(query_path) if query_path is not None else SEARCH_REVIEW_DIR / "eecs_queries.csv"

    if rebuild or not query_path.exists():
        queries_df = create_default_queries(query_path)
    else:
        queries_df = pd.read_csv(query_path)

    print(f"Queries loaded from: {query_path}")
    print(f"Queries: {len(queries_df)} queries")
    for _, row in queries_df.iterrows():
        print(f"  {row['query_id']}: {row['query_text']}")
    return queries_df


QUERY_PATH = SEARCH_REVIEW_DIR / "eecs_queries.csv"

# Keep False when using the reviewed starter CSVs in search_review/.
# Set True only when you intentionally want to regenerate the starter query list.
REBUILD_QUERIES = False

queries_df = load_premade_queries(QUERY_PATH, rebuild=REBUILD_QUERIES)


Queries loaded from: search_review\eecs_queries.csv
Queries: 14 queries
  q01: Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.
  q02: Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.
  q03: Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.
  q04: Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.
  q05: Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.
  q06: Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering gradua

In [7]:
QUERY_PATH = SEARCH_REVIEW_DIR / "eecs_queries.csv"

# Keep False when using the reviewed starter CSVs in search_review/.
# Set True only when you intentionally want to regenerate the starter query list.
REBUILD_QUERIES = False

queries_df = load_premade_queries(QUERY_PATH, rebuild=REBUILD_QUERIES)


Queries loaded from: search_review\eecs_queries.csv
Queries: 14 queries
  q01: Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.
  q02: Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.
  q03: Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.
  q04: Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.
  q05: Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.
  q06: Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering gradua

In [8]:
# Dense embedding model used by FAISS.
# BGE-M3 is the retrieval upgrade for this notebook.
# It improves the dense FAISS retrieval side; BM25 still handles the sparse keyword side.
MODEL_NAME = globals().get("MODEL_NAME", "BAAI/bge-m3")

# BGE-M3 is larger than MiniLM, so a smaller batch size is safer on laptops/CPU.
# Increase this if you have a strong GPU.
EMBED_BATCH_SIZE = globals().get(
    "EMBED_BATCH_SIZE",
    16 if "bge-m3" in MODEL_NAME.lower() else 64
)

print(f"Loading embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Embedding batch size: {EMBED_BATCH_SIZE}")


Loading embedding model: BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 1024
Embedding batch size: 16


In [9]:
def faiss_embedding_process(corpus_df):
    doc_ids = corpus_df['doc_id'].tolist()
    doc_texts = corpus_df['text'].astype(str).tolist()
    
    print(f"Encoding {len(doc_texts)} documents...")
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2 normalize for cosine similarity
    )
    
    print(f"Embedding shape: {doc_embeddings.shape}")
    print(f"Sample vector (first 10 dims): {doc_embeddings[0][:10]}")

    return doc_ids, doc_texts, doc_embeddings
    
    


In [10]:
doc_ids, doc_texts, doc_embeddings = faiss_embedding_process(active_corpus_df)


Encoding 577 documents...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Embedding shape: (577, 1024)
Sample vector (first 10 dims): [-0.04400833 -0.0446574   0.00981059 -0.00244941 -0.01062262  0.01916227
  0.01983805  0.02718064 -0.02253107  0.04992457]


In [11]:
def faiss_index_process(doc_embeddings):
    dim = doc_embeddings.shape[1]  # 384
    index = faiss.IndexFlatIP(dim)
    index.add(doc_embeddings.astype(np.float32))
    
    print(f"FAISS index built: {index.ntotal} vectors, dimension={dim}")

    return index


In [12]:
index = faiss_index_process(doc_embeddings)


FAISS index built: 577 vectors, dimension=1024


In [13]:
def dense_search(query_text, model, index, doc_ids, top_k=10):
    """
    Encode a query and return the top-k most similar documents.

    Returns: list of (doc_id, score, rank) tuples
    """
    # Encode the query with the SAME model and normalization
    query_vec = model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    # Search the FAISS index
    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append((doc_ids[idx], float(score), rank))

    return results


In [14]:
test_query = queries_df.iloc[0]
print(f"Query: {test_query['query_id']} — \"{test_query['query_text']}\"\n")

results = dense_search(test_query['query_text'], model, index, doc_ids, top_k=5)

for doc_id, score, rank in results:
    doc_text = active_corpus_df[active_corpus_df['doc_id'] == doc_id]['text'].values[0]
    preview = str(doc_text)[:200] + '...' if len(str(doc_text)) > 200 else str(doc_text)
    print(f"  Rank {rank} | Score: {score:.4f} | {doc_id}")
    print(f"    {preview}\n")


Query: q01 — "Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements."

  Rank 1 | Score: 0.7207 | pdf_0281_artificial-intelligence-certificate-program-application-jan6::chunk_001
    Document_Type = program_form_or_worksheet, Title = Artificial Intelligence Certificate Program Application Jan6, Course_Code = , Prefix = , Number = , Level = , Parent_Doc_ID = pdf_0281_artificial-int...

  Rank 2 | Score: 0.7199 | pdf_0513_professional-ai-certificate-program-application::chunk_001
    Document_Type = program_form_or_worksheet, Title = Professional Ai Certificate Program Application, Course_Code = , Prefix = , Number = , Level = , Parent_Doc_ID = pdf_0513_professional-ai-certificate...

  Rank 3 | Score: 0.7100 | pdf_0514_professional-certificate-ai-worksheet::chunk_001
    Document_Type = program_form_or_worksheet, Title = Professional Certificate Ai Worksheet, Course_Cod

In [15]:
TOP_K = 75
all_results = []

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    results = dense_search(qtext, model, index, doc_ids, top_k=TOP_K)
    for doc_id, score, rank in results:
        all_results.append({
            'query_id': qid,
            'doc_id': doc_id,
            'rank': rank,
            'score': round(score, 4),
        })

run_dense_df = pd.DataFrame(all_results)
run_dense_df.to_csv('outputs/run_dense.csv', index=False)

print(f"Dense retrieval results saved: {len(run_dense_df)} rows")
print(f"Queries processed: {run_dense_df['query_id'].nunique()}")
run_dense_df.head(20)


Dense retrieval results saved: 1050 rows
Queries processed: 14


,query_id,doc_id,rank,score
0,q01,pdf_0281_artificial-intelligence-certificate-p...,1,0.7207
1,q01,pdf_0513_professional-ai-certificate-program-a...,2,0.7199
2,q01,pdf_0514_professional-certificate-ai-worksheet...,3,0.7100
3,q01,pdf_0341_certificate-artificial--intelligence-...,4,0.7076
4,q01,pdf_0343_certificate-artificial-intelligence-p...,5,0.6876
5,q01,pdf_0342_certificate-artificial-intelligence--...,6,0.6865
6,q01,pdf_0278_artificial-inteligence-certificate-wo...,7,0.6691
7,q01,course_0025_CAP_6635::chunk_001,8,0.6553
8,q01,pdf_0510_newcertificate-artificial-intelligenc...,9,0.6542
9,q01,pdf_0280_artificial-intelligence-cert-program-...,10,0.6520


In [16]:
def tokenize_for_bm25(text):
    words = re.findall(r"[A-Za-z0-9]+", str(text).lower())
    return [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]


def bm25_tokenize_corpus(corpus_df):
    tokenized_corpus = [tokenize_for_bm25(doc) for doc in corpus_df["text"]]
    return tokenized_corpus


def bm25_corpus(tokenized_corpus):
    bm25 = BM25Okapi(tokenized_corpus, k1=2.4, b=1.0)
    return bm25


In [17]:
tokenized_corpus = bm25_tokenize_corpus(active_corpus_df)
bm25 = bm25_corpus(tokenized_corpus)


In [18]:
bm25_results = []
for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = tokenize_for_bm25(qrow['query_text'])
    scores = bm25.get_scores(qtext)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_indices, start=1):
        bm25_results.append({
            'query_id': qid,
            'doc_id': doc_ids[idx],
            'rank': rank,
            'score': round(float(scores[idx]), 4),
        })

run_bm25_df = pd.DataFrame(bm25_results)
run_bm25_df.to_csv('outputs/run_bm25.csv', index=False)
print(f"BM25 results: {len(run_bm25_df)} rows")
run_bm25_df.head(10)


BM25 results: 1050 rows


,query_id,doc_id,rank,score
0,q01,pdf_0513_professional-ai-certificate-program-a...,1,56.2046
1,q01,pdf_0514_professional-certificate-ai-worksheet...,2,54.9055
2,q01,pdf_0341_certificate-artificial--intelligence-...,3,42.9308
3,q01,pdf_0281_artificial-intelligence-certificate-p...,4,42.3594
4,q01,pdf_0515_professional-certificate-big-data-ana...,5,40.6248
5,q01,pdf_0516_professional-certificate-big-data-ana...,6,40.2413
6,q01,pdf_0483_minor--artificial-intelligence-applic...,7,38.8894
7,q01,course_0021_CAP_6629::chunk_001,8,38.5411
8,q01,course_0018_CAP_6619::chunk_001,9,38.5411
9,q01,pdf_0282_artificial-intelligence-minor-program...,10,37.3867


In [19]:
comparison_rows = []

for qid in queries_df['query_id']:
    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    for rank in range(1, 6):
        bm25_row = bm25_q[bm25_q['rank'] == rank]
        dense_row = dense_q[dense_q['rank'] == rank]

        comparison_rows.append({
            'query_id': qid,
            'rank': rank,
            'bm25_doc_id': bm25_row['doc_id'].values[0] if len(bm25_row) > 0 else '',
            'bm25_score': round(bm25_row['score'].values[0], 4) if len(bm25_row) > 0 else '',
            'dense_doc_id': dense_row['doc_id'].values[0] if len(dense_row) > 0 else '',
            'dense_score': round(dense_row['score'].values[0], 4) if len(dense_row) > 0 else '',
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('outputs/comparison.csv', index=False)

print("Side-by-side comparison saved to comparison.csv")
comparison_df.head(10)


Side-by-side comparison saved to comparison.csv


,query_id,rank,bm25_doc_id,bm25_score,dense_doc_id,dense_score
0,q01,1,pdf_0513_professional-ai-certificate-program-a...,56.2046,pdf_0281_artificial-intelligence-certificate-p...,0.7207
1,q01,2,pdf_0514_professional-certificate-ai-worksheet...,54.9055,pdf_0513_professional-ai-certificate-program-a...,0.7199
2,q01,3,pdf_0341_certificate-artificial--intelligence-...,42.9308,pdf_0514_professional-certificate-ai-worksheet...,0.7100
3,q01,4,pdf_0281_artificial-intelligence-certificate-p...,42.3594,pdf_0341_certificate-artificial--intelligence-...,0.7076
4,q01,5,pdf_0515_professional-certificate-big-data-ana...,40.6248,pdf_0343_certificate-artificial-intelligence-p...,0.6876
5,q02,1,pdf_0483_minor--artificial-intelligence-applic...,54.5321,pdf_0282_artificial-intelligence-minor-program...,0.7636
6,q02,2,pdf_0282_artificial-intelligence-minor-program...,53.1903,pdf_0483_minor--artificial-intelligence-applic...,0.7550
7,q02,3,pdf_0279_artificial-inteligence-minor-workshee...,44.6693,pdf_0279_artificial-inteligence-minor-workshee...,0.7189
8,q02,4,pdf_0342_certificate-artificial-intelligence--...,40.5448,pdf_0342_certificate-artificial-intelligence--...,0.7001
9,q02,5,pdf_0484_minor-in-business-application::chunk_001,39.9807,pdf_0281_artificial-intelligence-certificate-p...,0.6934


In [20]:
doc_lookup = dict(zip(active_corpus_df['doc_id'], active_corpus_df['text'].astype(str)))

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    print(f"\n{'='*80}")
    print(f"Query {qid}: \"{qtext}\"")
    print(f"{'='*80}")

    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    print(f"\n{'BM25 Results':<40} | {'Dense Retrieval Results':<40}")
    print(f"{'-'*40} | {'-'*40}")

    for rank in range(5):
        # BM25 side
        if rank < len(bm25_q):
            b = bm25_q.iloc[rank]
            b_text = doc_lookup.get(b['doc_id'], '')[:60]
            bm25_str = f"#{b['rank']} {b['doc_id']} ({b['score']:.2f}) {b_text}"
        else:
            bm25_str = ""

        # Dense side
        if rank < len(dense_q):
            d = dense_q.iloc[rank]
            d_text = doc_lookup.get(d['doc_id'], '')[:60]
            dense_str = f"#{d['rank']} {d['doc_id']} ({d['score']:.4f}) {d_text}"
        else:
            dense_str = ""

        print(f"{bm25_str:<40} | {dense_str:<40}")

    # Overlap analysis
    bm25_docs = set(bm25_q['doc_id'])
    dense_docs = set(dense_q['doc_id'])
    overlap = bm25_docs & dense_docs
    print(f"\nOverlap: {len(overlap)} / 5 docs appear in both top-5")
    if overlap:
        print(f"  Shared docs: {overlap}")
    print(f"  BM25-only: {bm25_docs - dense_docs}")
    print(f"  Dense-only: {dense_docs - bm25_docs}")



Query q01: "Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements."

BM25 Results                             | Dense Retrieval Results                 
---------------------------------------- | ----------------------------------------
#1 pdf_0513_professional-ai-certificate-program-application::chunk_001 (56.20) Document_Type = program_form_or_worksheet, Title = Professio | #1 pdf_0281_artificial-intelligence-certificate-program-application-jan6::chunk_001 (0.7207) Document_Type = program_form_or_worksheet, Title = Artificia
#2 pdf_0514_professional-certificate-ai-worksheet::chunk_001 (54.91) Document_Type = program_form_or_worksheet, Title = Professio | #2 pdf_0513_professional-ai-certificate-program-application::chunk_001 (0.7199) Document_Type = program_form_or_worksheet, Title = Professio
#3 pdf_0341_certificate-artificial--intelligence-program-application::chun

In [21]:
def rrf_process(bm25_df, dense_df, k=300, bm25_weight=0.4, dense_weight=3.0):
    fused_results = []

    bm_ids = set(bm25_df["query_id"])
    dense_ids = set(dense_df["query_id"])
    query_ids = sorted(bm_ids.union(dense_ids))

    for qid in query_ids:
        rrf_scores = {}

        bm25_q = bm25_df[bm25_df["query_id"] == qid]
        dense_q = dense_df[dense_df["query_id"] == qid]

        # BM25 contribution
        for _, row in bm25_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                bm25_weight / (k + rank)
            )

        # Dense contribution
        for _, row in dense_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                dense_weight / (k + rank)
            )

        ranked_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        for final_rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            fused_results.append({
                "query_id": qid,
                "doc_id": doc_id,
                "rrf_score": round(score, 6),
                "rank": final_rank
            })

    fused_df = pd.DataFrame(fused_results)
    return fused_df

    


In [22]:
# Build and save the optimized weighted-RRF results.
# The separate FLAN-T5 result-summary stage has been removed.
run_rrf_df = rrf_process(
    run_bm25_df,
    run_dense_df,
    k=300,
    bm25_weight=0.4,
    dense_weight=3.0
)

run_rrf_df.to_csv("outputs/run_rrf.csv", index=False)

print(f"Optimized RRF results: {len(run_rrf_df)} rows")
run_rrf_df.head(10)


Optimized RRF results: 1571 rows


,query_id,doc_id,rrf_score,rank
0,q01,pdf_0281_artificial-intelligence-certificate-p...,0.011283,1
1,q01,pdf_0513_professional-ai-certificate-program-a...,0.011263,2
2,q01,pdf_0514_professional-certificate-ai-worksheet...,0.011225,3
3,q01,pdf_0341_certificate-artificial--intelligence-...,0.011189,4
4,q01,pdf_0343_certificate-artificial-intelligence-p...,0.011106,5
5,q01,pdf_0342_certificate-artificial-intelligence--...,0.011070,6
6,q01,pdf_0278_artificial-inteligence-certificate-wo...,0.011050,7
7,q01,pdf_0510_newcertificate-artificial-intelligenc...,0.010943,8
8,q01,pdf_0282_artificial-intelligence-minor-program...,0.010937,9
9,q01,pdf_0280_artificial-intelligence-cert-program-...,0.010908,10


In [23]:
qrels_path = SEARCH_REVIEW_DIR / "eecs_qrels.csv"
qrels_available = qrels_path.exists()

if qrels_available:
    qrels_original_df = pd.read_csv(qrels_path).fillna("")
    qrels_df = qrels_original_df.copy()

    # If retrieval is chunked, qrels written for original source documents need to be expanded
    # so evaluation compares retrieved chunk IDs against chunk-level relevance labels.
    if globals().get("RETRIEVAL_IS_CHUNKED", False) and "parent_doc_id" in active_corpus_df.columns:
        chunk_map = active_corpus_df[["doc_id", "parent_doc_id"]].copy()
        chunk_map = chunk_map.rename(columns={"doc_id": "chunk_doc_id"})

        expanded_qrels = qrels_original_df.merge(
            chunk_map,
            left_on="doc_id",
            right_on="parent_doc_id",
            how="inner"
        )

        if not expanded_qrels.empty:
            qrels_df = (
                expanded_qrels[["query_id", "chunk_doc_id", "relevance"]]
                .rename(columns={"chunk_doc_id": "doc_id"})
                .drop_duplicates()
                .reset_index(drop=True)
            )
            print(
                f"Expanded qrels from {len(qrels_original_df)} document-level rows "
                f"to {len(qrels_df)} chunk-level rows for evaluation."
            )
        else:
            print("Warning: qrels could not be expanded to chunk IDs. Metrics may be zero if IDs do not overlap.")
else:
    qrels_original_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])
    qrels_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])
    print("No search_review/eecs_qrels.csv file was found. Skipping supervised retrieval evaluation metrics.")
    print("Retrieval still works; qrels are only needed for P@K, MAP, and nDCG scoring.")


def precision_at_k(ranked_doc_ids, rel_map, k=10):
    top = ranked_doc_ids[:k]
    if len(top) == 0:
        return 0.0
    hits = sum(1 for d in top if rel_map.get(d, 0) > 0)  # relevance 1 or 2 counts as relevant
    return hits / len(top)

# Build relevance lookup: qid -> {doc_id: relevance}
rel_by_q = {}
if qrels_available:
    for qid, g in qrels_df.groupby("query_id"):
        rel_by_q[str(qid)] = {
            str(r.doc_id): int(r.relevance)
            for r in g.itertuples(index=False)
        }


def evaluate_p10(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping P@{k} for {run_name}; qrels are not available.")
        return 0.0

    p10_values = []

    print(f"\nP@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        p10 = precision_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        p10_values.append(p10)
        print(f"{qid}: {p10:.3f}")

    mean_p10 = float(np.mean(p10_values)) if p10_values else 0.0
    print(f"Mean P@{k} for {run_name}: {mean_p10:.3f}")
    return mean_p10

# Evaluate BM25
mean_p10_bm25 = evaluate_p10(run_bm25_df, "BM25", k=10)

# Evaluate FAISS / Dense retrieval
mean_p10_faiss = evaluate_p10(run_dense_df, "FAISS Dense Search", k=10)

# Evaluate RRF
mean_p10_rrf = evaluate_p10(run_rrf_df, "RRF Fusion", k=10)


Expanded qrels from 388 document-level rows to 181 chunk-level rows for evaluation.

P@10 per query (BM25):
q01: 0.300
q02: 0.400
q03: 0.400
q04: 0.000
q05: 0.000
q06: 0.000
q07: 0.100
q08: 0.300
q09: 0.400
q10: 0.100
q11: 0.900
q12: 0.000
q13: 0.800
q14: 0.400
Mean P@10 for BM25: 0.293

P@10 per query (FAISS Dense Search):
q01: 0.600
q02: 0.600
q03: 0.300
q04: 0.000
q05: 0.100
q06: 0.200
q07: 0.200
q08: 0.500
q09: 0.600
q10: 0.500
q11: 0.900
q12: 0.100
q13: 0.900
q14: 0.000
Mean P@10 for FAISS Dense Search: 0.393

P@10 per query (RRF Fusion):
q01: 0.700
q02: 0.600
q03: 0.300
q04: 0.000
q05: 0.100
q06: 0.200
q07: 0.200
q08: 0.400
q09: 0.600
q10: 0.500
q11: 0.900
q12: 0.200
q13: 0.900
q14: 0.100
Mean P@10 for RRF Fusion: 0.407


In [24]:
print("\n" + "=" * 50)
print("Mean P@10 Summary")
print("=" * 50)
print(f"BM25:              {mean_p10_bm25:.3f}")
print(f"FAISS Dense:       {mean_p10_faiss:.3f}")
print(f"RRF Fusion:        {mean_p10_rrf:.3f}")



Mean P@10 Summary
BM25:              0.293
FAISS Dense:       0.393
RRF Fusion:        0.407


In [25]:
def average_precision(ranked_doc_ids, rel_map):
    # AP uses binary relevance: treat relevance 1 or 2 as relevant
    total_relevant = sum(1 for v in rel_map.values() if v > 0)
    if total_relevant == 0:
        return 0.0

    hits = 0
    ap_sum = 0.0

    for i, d in enumerate(ranked_doc_ids, start=1):
        if rel_map.get(d, 0) > 0:
            hits += 1
            ap_sum += hits / i

    return ap_sum / total_relevant


def evaluate_map(run_df, run_name):
    if not qrels_available:
        print(f"Skipping MAP for {run_name}; qrels are not available.")
        return 0.0

    ap_values = []

    print(f"\nAverage Precision (AP) per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ap = average_precision(ranked, rel_by_q.get(str(qid), {}))
        ap_values.append(ap)
        print(f"{qid}: {ap:.3f}")

    map_score = float(np.mean(ap_values)) if len(ap_values) > 0 else 0.0
    print(f"\nMAP ({run_name}): {map_score:.3f}")
    return map_score


# ---- BM25 ----
map_bm25 = evaluate_map(run_bm25_df, "BM25")

# ---- FAISS / Dense ----
map_faiss = evaluate_map(run_dense_df, "FAISS Dense Search")

# ---- RRF ----
map_rrf = evaluate_map(run_rrf_df, "RRF Fusion")


print("\n" + "=" * 50)
print("MAP Summary")
print("=" * 50)
print(f"BM25:         {map_bm25:.3f}")
print(f"FAISS Dense:  {map_faiss:.3f}")
print(f"RRF Fusion:   {map_rrf:.3f}")



Average Precision (AP) per query (BM25):
q01: 0.289
q02: 0.473
q03: 0.476
q04: 0.000
q05: 0.052
q06: 0.062
q07: 0.222
q08: 0.363
q09: 0.295
q10: 0.266
q11: 0.709
q12: 0.059
q13: 0.423
q14: 0.306

MAP (BM25): 0.285

Average Precision (AP) per query (FAISS Dense Search):
q01: 0.509
q02: 0.664
q03: 0.387
q04: 0.009
q05: 0.038
q06: 0.082
q07: 0.188
q08: 0.277
q09: 0.480
q10: 0.529
q11: 0.592
q12: 0.124
q13: 0.681
q14: 0.010

MAP (FAISS Dense Search): 0.327

Average Precision (AP) per query (RRF Fusion):
q01: 0.529
q02: 0.673
q03: 0.412
q04: 0.008
q05: 0.064
q06: 0.093
q07: 0.240
q08: 0.349
q09: 0.480
q10: 0.495
q11: 0.707
q12: 0.170
q13: 0.678
q14: 0.058

MAP (RRF Fusion): 0.354

MAP Summary
BM25:         0.285
FAISS Dense:  0.327
RRF Fusion:   0.354


In [26]:
def dcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = 0.0
    for i, d in enumerate(ranked_doc_ids[:k], start=1):
        gain = rel_map.get(d, 0)  # graded relevance: 0, 1, 2
        dcg += gain / math.log2(i + 1)
    return dcg


def ndcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = dcg_at_k(ranked_doc_ids, rel_map, k)

    # Ideal DCG: sort true relevance scores from highest to lowest
    ideal_gains = sorted(rel_map.values(), reverse=True)
    idcg = 0.0
    for i, gain in enumerate(ideal_gains[:k], start=1):
        idcg += gain / math.log2(i + 1)

    return (dcg / idcg) if idcg > 0 else 0.0


def evaluate_ndcg(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping nDCG@{k} for {run_name}; qrels are not available.")
        return 0.0

    ndcg_values = []

    print(f"\nnDCG@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ndcg = ndcg_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        ndcg_values.append(ndcg)
        print(f"{qid}: {ndcg:.3f}")

    mean_ndcg = float(np.mean(ndcg_values)) if len(ndcg_values) > 0 else 0.0
    print(f"\nMean nDCG@{k} ({run_name}): {mean_ndcg:.3f}")
    return mean_ndcg

# ---- BM25 ----
mean_ndcg_bm25 = evaluate_ndcg(run_bm25_df, "BM25", k=30)

# ---- FAISS / Dense ----
mean_ndcg_faiss = evaluate_ndcg(run_dense_df, "FAISS Dense Search", k=30)

# ---- RRF ----
mean_ndcg_rrf = evaluate_ndcg(run_rrf_df, "RRF Fusion", k=30)

print("\n" + "=" * 50)
print("nDCG@30 Summary")
print("=" * 50)
print(f"BM25:         {mean_ndcg_bm25:.3f}")
print(f"FAISS Dense:  {mean_ndcg_faiss:.3f}")
print(f"RRF Fusion:   {mean_ndcg_rrf:.3f}")



nDCG@30 per query (BM25):
q01: 0.546
q02: 0.650
q03: 0.633
q04: 0.000
q05: 0.041
q06: 0.131
q07: 0.300
q08: 0.486
q09: 0.435
q10: 0.392
q11: 0.759
q12: 0.120
q13: 0.672
q14: 0.479

Mean nDCG@30 (BM25): 0.403

nDCG@30 per query (FAISS Dense Search):
q01: 0.779
q02: 0.852
q03: 0.545
q04: 0.000
q05: 0.203
q06: 0.248
q07: 0.352
q08: 0.494
q09: 0.597
q10: 0.705
q11: 0.725
q12: 0.270
q13: 0.776
q14: 0.053

Mean nDCG@30 (FAISS Dense Search): 0.471

nDCG@30 per query (RRF Fusion):
q01: 0.786
q02: 0.851
q03: 0.635
q04: 0.000
q05: 0.209
q06: 0.242
q07: 0.413
q08: 0.527
q09: 0.593
q10: 0.658
q11: 0.798
q12: 0.409
q13: 0.790
q14: 0.105

Mean nDCG@30 (RRF Fusion): 0.501

nDCG@30 Summary
BM25:         0.403
FAISS Dense:  0.471
RRF Fusion:   0.501


Final Interactive Hybrid RAG Loop — Quality-Guarded Version
¶
This final block keeps the retrieval and answer-generation steps separated. The user's original question is preserved for the LLM-facing prompt, while a retrieval-optimized version is used only for BM25 + FAISS + RRF. The retrieved chunks are compressed into query-relevant background facts, inserted into the Hugging Face prompt, and hidden from normal output. Only the final contextualized answer is displayed.
This version also includes answer-quality guards to reduce document-title copying, repeated-token failures, and irrelevant context leakage.

In [27]:
# ============================================================
# Final Interactive Hybrid RAG Loop
# Keyword-Gated Retrieval + Conversation Continuation Version
# ============================================================
# Paste this whole block at the end of the notebook after:
# - corpus_df / active_corpus_df have been created
# - BM25 has been built as bm25
# - FAISS has been built as index
# - doc_ids has been created for the active retrieval dataframe
# - dense_search(), tokenize_for_bm25(), and rrf_process() exist
# - the dense embedding model variable is named model
#
# Behavior:
# 1. User enters a query.
# 2. If the query contains major EECS keywords, fresh BM25 + FAISS + RRF retrieval runs.
# 3. If no major EECS keywords are found and previous retrieved context exists,
#    retrieval is skipped and the previous context is reused.
# 4. The LLM receives the original question plus hidden retrieved context.
# 5. The user only sees the final contextualized answer.

import os

# ----------------------------
# RAG settings
# ----------------------------
RAG_TOP_K = 10
RAG_RETRIEVAL_POOL = 75
RAG_RRF_K = 300
# Tested weighted-RRF configuration: dense retrieval is emphasized while BM25 still supports exact terms.
RAG_BM25_WEIGHT = 0.4
RAG_DENSE_WEIGHT = 3.0
# RAG_MAX_CONTEXT_CHARS = 3500
# RAG_MAX_SNIPPET_CHARS = 520
# RAG_MAX_INPUT_TOKENS = 1536
# RAG_MAX_NEW_TOKENS = 180
# RAG_DEBUG = False
RAG_MAX_INPUT_TOKENS = 2048
RAG_MAX_CONTEXT_CHARS = 5500
RAG_MAX_SNIPPET_CHARS = 600
RAG_MAX_NEW_TOKENS = 220
RAG_DEBUG = False


# Recommended options:
#   Qwen/Qwen2.5-3B-Instruct      -> stronger answers, heavier
#   Qwen/Qwen2.5-1.5B-Instruct    -> lighter/faster
#   Qwen/Qwen2.5-0.5B-Instruct    -> very small, weaker
#   google/flan-t5-base           -> legacy seq2seq option
RAG_MODEL_NAME = globals().get("RAG_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")

# Optional Hugging Face token.
# Either set HF_TOKEN in an earlier cell or set it as an environment variable.
HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", None))
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN


# ----------------------------
# Answer model loading
# ----------------------------
def is_seq2seq_rag_model(model_name):
    name = str(model_name).lower()
    return "t5" in name or "flan" in name


def load_rag_generation_model(model_name):
    print(f"Loading Hugging Face RAG answer model: {model_name}")

    tokenizer_kwargs = {"trust_remote_code": True}
    if HF_TOKEN:
        tokenizer_kwargs["token"] = HF_TOKEN

    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)

    model_kwargs = {"trust_remote_code": True}
    if HF_TOKEN:
        model_kwargs["token"] = HF_TOKEN

    if is_seq2seq_rag_model(model_name):
        model_obj = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
        model_kind = "seq2seq"
    else:
        # Use dtype instead of deprecated torch_dtype.
        if torch.cuda.is_available():
            model_kwargs.update({"device_map": "auto", "dtype": torch.float16})
        else:
            model_kwargs.update({"dtype": torch.float32})
        model_obj = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        model_kind = "causal"

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    if not torch.cuda.is_available():
        model_obj = model_obj.to("cpu")

    model_obj.eval()
    print(f"RAG answer model loaded as: {model_kind}")
    return tokenizer, model_obj, model_kind


rag_tokenizer, rag_model, RAG_MODEL_KIND = load_rag_generation_model(RAG_MODEL_NAME)


# ----------------------------
# Query utilities
# ----------------------------
RAG_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "could", "do", "does",
    "for", "from", "how", "i", "in", "is", "it", "me", "my", "need", "needed", "of",
    "on", "or", "should", "the", "their", "there", "this", "to", "what", "when", "where",
    "which", "who", "will", "with", "would", "you", "your", "about", "related", "information",
    "student"
}


def rag_normalize_text(text):
    return " ".join(str(text).replace("\x00", " ").split())


def rag_terms(text):
    words = re.findall(r"[A-Za-z][A-Za-z0-9]+", str(text).lower())
    return [w for w in words if w not in RAG_STOPWORDS and len(w) > 2]


def classify_rag_intent(original_query):
    q = str(original_query).lower()

    if any(t in q for t in ["recommend", "class", "classes", "course", "courses", "take", "fall", "spring", "summer", "senior"]):
        return "course_recommendation"

    if "certificate" in q or "certification" in q:
        return "certificate_requirements"

    if any(t in q for t in ["credit", "credits", "bachelor", "bachelors", "b.s.", "bs ", "undergraduate degree", "degree requirements"]):
        return "degree_requirements"

    if any(t in q for t in ["chair", "director", "advisor", "coordinator", "dean", "acting chair"]):
        return "person_lookup"

    if any(t in q for t in ["form", "worksheet", "application", "audit", "pos", "plan of study"]):
        return "forms"

    return "general"


EECS_RETRIEVAL_EXPANSIONS_BY_INTENT = {
    "course_recommendation": [
        "course listing", "course offerings", "undergraduate", "senior", "4000", "3000", "fall",
        "CAP", "CDA", "CEN", "CIS", "COP", "COT", "EEL", "EEE", "prerequisite", "credits"
    ],
    "certificate_requirements": [
        "certificate", "requirements", "required courses", "credits", "worksheet", "program sheet",
        "application", "big data analytics", "artificial intelligence", "cybersecurity", "data science"
    ],
    "degree_requirements": [
        "degree requirements", "program sheet", "worksheet", "bachelor of science", "BS", "BSEE",
        "undergraduate", "total credits", "required credits", "electrical engineering", "computer science"
    ],
    "person_lookup": [
        "acting chair", "department chair", "chair", "director", "faculty",
        "computer science engineering electrical engineering"
    ],
    "forms": [
        "form", "forms", "worksheet", "application", "degree audit", "plan of study", "POS", "requirements"
    ],
    "general": [
        "computer science", "electrical engineering", "department", "course", "program", "requirements"
    ]
}

EECS_KEYWORD_EXPANSIONS = {
    "ai": ["artificial intelligence", "machine learning"],
    "artificial intelligence": ["AI", "machine learning"],
    "ml": ["machine learning", "artificial intelligence"],
    "big data": ["big data analytics", "data analytics", "data science"],
    "cyber": ["cybersecurity", "cyber security", "security"],
    "cybersecurity": ["cyber security", "security"],
    "computer science": ["CS", "CSE", "COP", "COT", "CAP", "CEN", "computer science"],
    "computer engineering": ["computer engineering", "CpE", "CDA", "EEE", "EEL"],
    "electrical engineering": ["electrical engineering", "EE", "EEL", "EEE", "BSEE"],
    "bachelors": ["bachelor", "bachelor of science", "BS", "undergraduate", "total credits"],
    "bachelor": ["bachelor of science", "BS", "undergraduate", "total credits"],
    "credits": ["credit hours", "total credits", "required credits"],
    "fall": ["fall semester", "course offerings", "schedule"],
    "senior": ["4000", "3000", "upper division", "undergraduate"],
}


def improve_query_for_hybrid_search(original_query, max_extra_terms=24):
    """Expand the query for retrieval only. The original query is still used in the LLM prompt."""
    original_query = str(original_query).strip()
    lowered = original_query.lower()
    intent = classify_rag_intent(original_query)
    extra_terms = []

    for term in EECS_RETRIEVAL_EXPANSIONS_BY_INTENT.get(intent, []):
        if term.lower() not in lowered and term not in extra_terms:
            extra_terms.append(term)

    for trigger, expansions in EECS_KEYWORD_EXPANSIONS.items():
        if trigger in lowered:
            for term in expansions:
                if term.lower() not in lowered and term not in extra_terms:
                    extra_terms.append(term)

    extra_terms = extra_terms[:max_extra_terms]
    return original_query if not extra_terms else original_query + " " + " ".join(extra_terms)


# ----------------------------
# Keyword gate for retrieval refresh
# ----------------------------
MAJOR_EECS_KEYWORDS = {
    # Department/program names
    "computer science", "cs", "cse", "electrical engineering", "ee", "computer engineering", "eecs", "engineering",

    # Course prefixes
    "cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee", "egm", "egn",

    # Academic planning terms
    "course", "courses", "class", "classes", "credit", "credits", "prerequisite", "prerequisites",
    "requirement", "requirements", "degree", "bachelor", "bachelors", "undergraduate", "graduate",
    "masters", "ms", "phd", "doctoral", "senior", "junior", "fall", "spring", "summer", "semester", "schedule",

    # Forms/certificates
    "form", "forms", "worksheet", "worksheets", "application", "plan of study", "pos", "degree audit", "certificate",

    # Topics
    "artificial intelligence", "ai", "machine learning", "ml", "data science", "big data", "big data analytics",
    "cybersecurity", "cyber security", "security", "software engineering", "database", "databases", "networks",
    "embedded", "robotics", "signals", "circuits", "systems", "programming", "algorithms",

    # People/department roles
    "chair", "acting chair", "director", "advisor", "coordinator", "faculty", "professor", "dean"
}

FOLLOWUP_CUES = {
    "that", "those", "these", "them", "it", "they", "which", "what about", "how about", "why",
    "explain", "continue", "more", "also", "compare", "rank", "best", "better", "easier", "harder",
    "recommend", "summarize", "clarify", "second", "third", "first", "last"
}


def query_has_major_eecs_keyword(query_text):
    """Return True if the query appears to introduce a new EECS retrieval topic."""
    q = str(query_text).lower().strip()
    if not q:
        return False

    for keyword in MAJOR_EECS_KEYWORDS:
        if keyword in q:
            return True

    # Course-code style match: COP 3530, COT3100, EEL 3111, CAP4611, etc.
    if re.search(r"\b(cap|cda|cen|cis|cop|cot|eel|eee|egm|egn)\s*\d{4}\b", q, flags=re.IGNORECASE):
        return True

    return False


def query_looks_like_followup(query_text):
    q = str(query_text).lower().strip()
    if not q:
        return False
    return any(cue in q for cue in FOLLOWUP_CUES)


def should_refresh_retrieval_for_query(query_text):
    """If major EECS keywords are present, refresh retrieval; otherwise reuse last context."""
    return query_has_major_eecs_keyword(query_text)


# ----------------------------
# Retrieval helpers
# ----------------------------
def bm25_search_custom(query_text, bm25_model, doc_ids, top_k=100):
    query_tokens = tokenize_for_bm25(query_text)
    scores = bm25_model.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        (doc_ids[idx], float(scores[idx]), rank)
        for rank, idx in enumerate(top_indices, start=1)
    ]


def hybrid_search_custom_query(
    retrieval_query_text,
    top_k=RAG_TOP_K,
    retrieval_pool=RAG_RETRIEVAL_POOL,
    rrf_k=RAG_RRF_K,
    bm25_weight=RAG_BM25_WEIGHT,
    dense_weight=RAG_DENSE_WEIGHT
):
    custom_qid = "custom_query"

    dense_results = dense_search(
        query_text=retrieval_query_text,
        model=model,
        index=index,
        doc_ids=doc_ids,
        top_k=retrieval_pool
    )

    dense_df = pd.DataFrame([
        {"query_id": custom_qid, "doc_id": doc_id, "rank": rank, "score": score}
        for doc_id, score, rank in dense_results
    ])

    bm25_results = bm25_search_custom(
        query_text=retrieval_query_text,
        bm25_model=bm25,
        doc_ids=doc_ids,
        top_k=retrieval_pool
    )

    bm25_df = pd.DataFrame([
        {"query_id": custom_qid, "doc_id": doc_id, "rank": rank, "score": score}
        for doc_id, score, rank in bm25_results
    ])

    fused_df = rrf_process(
        bm25_df,
        dense_df,
        k=rrf_k,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight
    )

    bm25_rank_lookup = bm25_df.set_index("doc_id")["rank"].to_dict()
    bm25_score_lookup = bm25_df.set_index("doc_id")["score"].to_dict()
    dense_rank_lookup = dense_df.set_index("doc_id")["rank"].to_dict()
    dense_score_lookup = dense_df.set_index("doc_id")["score"].to_dict()

    fused_df["bm25_rank"] = fused_df["doc_id"].map(bm25_rank_lookup)
    fused_df["bm25_score"] = fused_df["doc_id"].map(bm25_score_lookup)
    fused_df["dense_rank"] = fused_df["doc_id"].map(dense_rank_lookup)
    fused_df["dense_score"] = fused_df["doc_id"].map(dense_score_lookup)

    return fused_df.sort_values("rank").reset_index(drop=True)


# ----------------------------
# Context compression
# ----------------------------
def get_search_dataframe():
    return globals().get("active_corpus_df", corpus_df)


def get_document_record(doc_id):
    search_df = get_search_dataframe()
    matches = search_df.loc[search_df["doc_id"].astype(str) == str(doc_id)]

    if matches.empty and "parent_doc_id" in search_df.columns:
        matches = search_df.loc[search_df["parent_doc_id"].astype(str) == str(doc_id)]

    if matches.empty:
        return {"doc_id": str(doc_id), "text": "", "chunk_text": ""}

    return matches.iloc[0].to_dict()


def get_record_value(record, *names):
    for name in names:
        value = str(record.get(name, "")).strip()
        if value and value.lower() not in {"nan", "none", "not found"}:
            return value
    return ""


def parse_rag_document(doc_id):
    record = get_document_record(doc_id)

    content = (
        get_record_value(record, "chunk_text")
        or get_record_value(record, "content", "Content")
        or get_record_value(record, "text")
    )

    return {
        "doc_id": str(doc_id),
        "parent_doc_id": get_record_value(record, "parent_doc_id") or str(doc_id),
        "chunk_index": get_record_value(record, "chunk_index"),
        "content": rag_normalize_text(content),
        "title": get_record_value(record, "title", "Title"),
        "document_type": get_record_value(record, "document_type", "Document_Type"),
        "course_code": get_record_value(record, "course_code", "Course_Code"),
        "prefix": get_record_value(record, "prefix", "Prefix"),
        "number": get_record_value(record, "number", "Number"),
        "level": get_record_value(record, "level", "Level"),
        "file_name": get_record_value(record, "file_name", "File_Name"),
        "source": get_record_value(record, "linked_file_or_page", "source_page", "Source_Page"),
    }


def split_into_candidate_sentences(text):
    text = rag_normalize_text(text)
    rough = re.split(r"(?<=[.!?])\s+|\s{2,}|\s+[•●]\s+", text)

    sentences = []
    for part in rough:
        part = part.strip(" -\t\n\r")
        if 25 <= len(part) <= 520:
            sentences.append(part)
        elif len(part) > 520:
            for i in range(0, len(part), 360):
                chunk = part[i:i + 520].strip()
                if len(chunk) >= 25:
                    sentences.append(chunk)

    return sentences


def intent_keywords(intent):
    return {
        "course_recommendation": [
            "course", "prerequisite", "credits", "undergraduate", "senior", "fall", "3000", "4000",
            "cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee"
        ],
        "certificate_requirements": [
            "certificate", "requirements", "required", "credits", "course", "courses", "complete", "program",
            "worksheet", "application"
        ],
        "degree_requirements": [
            "degree", "requirements", "credits", "credit", "bachelor", "undergraduate", "program",
            "curriculum", "total", "engineering"
        ],
        "person_lookup": ["chair", "acting", "director", "coordinator", "advisor", "faculty", "professor"],
        "forms": ["form", "worksheet", "application", "audit", "plan", "study", "requirements"],
        "general": []
    }.get(intent, [])


def score_text_for_query(text, original_query, retrieval_query, intent):
    lowered = str(text).lower()
    q_terms = set(rag_terms(original_query)) | set(rag_terms(retrieval_query))
    score = 0.0

    for term in q_terms:
        if term in lowered:
            score += 2.0

    for term in intent_keywords(intent):
        if str(term).lower() in lowered:
            score += 1.5

    # Penalize irrelevant healthcare carryover/noisy text.
    for term in ["patient", "diagnosis", "procedure", "medication", "clinical", "heart disease"]:
        if term in lowered:
            score -= 8.0

    # Penalize repeated-word PDF extraction failures.
    words = rag_terms(lowered)
    if len(words) >= 30:
        most_common_count = max([words.count(w) for w in set(words)] or [0])
        if most_common_count / max(len(words), 1) > 0.18:
            score -= 6.0

    return score


def select_relevant_snippet(doc, original_query, retrieval_query, intent, max_chars=RAG_MAX_SNIPPET_CHARS):
    candidate_text = doc.get("content", "")
    sentences = split_into_candidate_sentences(candidate_text)

    if not sentences:
        return rag_normalize_text(candidate_text)[:max_chars]

    scored = [(score_text_for_query(sent, original_query, retrieval_query, intent), sent) for sent in sentences]
    scored.sort(key=lambda x: x[0], reverse=True)

    selected = []
    total_chars = 0

    for score, sent in scored[:8]:
        if score <= 0 and selected:
            continue
        if sent in selected:
            continue
        if total_chars + len(sent) > max_chars and selected:
            continue

        selected.append(sent)
        total_chars += len(sent)

        if total_chars >= max_chars:
            break

    if not selected:
        selected = [sentences[0][:max_chars]]

    return rag_normalize_text(" ".join(selected))[:max_chars].rstrip()


def document_intent_boost(doc, original_query, retrieval_query, intent):
    combined = " ".join([
        doc.get("title", ""),
        doc.get("document_type", ""),
        doc.get("course_code", ""),
        doc.get("file_name", ""),
        doc.get("source", ""),
        doc.get("content", "")[:1200]
    ]).lower()

    boost = 0.0

    if intent == "course_recommendation":
        if "course" in combined:
            boost += 6
        if any(prefix in combined for prefix in ["cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee"]):
            boost += 3
        if any(level in combined for level in ["undergraduate", "3000", "4000", "senior"]):
            boost += 3
        if any(term in combined for term in ["phd", "doctoral"]):
            boost -= 5

    elif intent == "certificate_requirements":
        if "certificate" in combined:
            boost += 8
        if any(term in combined for term in ["requirement", "required", "credits", "worksheet", "application"]):
            boost += 4
        if "big data" in original_query.lower() and "big data" in combined:
            boost += 8

    elif intent == "degree_requirements":
        if any(term in combined for term in ["degree requirements", "program sheet", "worksheet", "curriculum"]):
            boost += 6
        if any(term in original_query.lower() for term in ["electrical", "ee"]):
            if any(term in combined for term in ["electrical engineering", "bsee", "eel", "eee"]):
                boost += 8
            if any(term in combined for term in ["computer science", "mscs", "phd"]):
                boost -= 4
        if any(term in combined for term in ["bachelor", "undergraduate", "total credits", "credits"]):
            boost += 5
        if any(term in combined for term in ["phd", "doctoral"]):
            boost -= 8

    elif intent == "person_lookup":
        if any(term in combined for term in ["chair", "acting chair", "faculty", "director"]):
            boost += 6
        if any(term in combined for term in ["syllabus", "homework", "assignment", "phd phd"]):
            boost -= 6

    elif intent == "forms":
        if any(term in combined for term in ["form", "worksheet", "application", "audit", "plan of study"]):
            boost += 6

    return boost


def get_fused_base_score(row):
    for col in ["rrf_score", "score", "fusion_score"]:
        if col in row and pd.notna(row.get(col)):
            try:
                return float(row.get(col))
            except Exception:
                pass
    return 0.0


def rerank_fused_results(fused_df, original_query, retrieval_query, top_k=RAG_TOP_K):
    intent = classify_rag_intent(original_query)
    reranked_rows = []

    for _, row in fused_df.head(RAG_RETRIEVAL_POOL).iterrows():
        doc_id = str(row["doc_id"])
        doc = parse_rag_document(doc_id)

        base_score = get_fused_base_score(row) * 100.0
        text_score = score_text_for_query(
            " ".join([
                doc.get("title", ""),
                doc.get("file_name", ""),
                doc.get("course_code", ""),
                doc.get("content", "")[:2500]
            ]),
            original_query,
            retrieval_query,
            intent
        )
        boost = document_intent_boost(doc, original_query, retrieval_query, intent)

        new_row = row.to_dict()
        new_row["rag_rerank_score"] = base_score + text_score + boost
        new_row["rag_intent"] = intent
        new_row["title"] = doc.get("title", "")
        new_row["course_code"] = doc.get("course_code", "")
        new_row["document_type"] = doc.get("document_type", "")
        new_row["file_name"] = doc.get("file_name", "")
        reranked_rows.append(new_row)

    reranked_df = pd.DataFrame(reranked_rows)

    if reranked_df.empty:
        return fused_df.head(top_k).copy()

    reranked_df = reranked_df.sort_values("rag_rerank_score", ascending=False).head(top_k).reset_index(drop=True)
    reranked_df["rank"] = range(1, len(reranked_df) + 1)
    return reranked_df


def build_fact_line(rank, doc, snippet):
    parts = []

    if doc.get("course_code"):
        parts.append(f"course {doc.get('course_code')}")
    if doc.get("title"):
        parts.append(f"titled {doc.get('title')}")
    if doc.get("level"):
        parts.append(f"level {doc.get('level')}")
    if doc.get("document_type"):
        parts.append(f"source type {doc.get('document_type')}")

    descriptor = ", ".join(parts)

    if descriptor:
        return f"Fact {rank}: In a department source for {descriptor}, the relevant information says: {snippet}"

    return f"Fact {rank}: {snippet}"


def build_hidden_rag_context(reranked_df, original_query, retrieval_query, max_total_chars=RAG_MAX_CONTEXT_CHARS):
    intent = classify_rag_intent(original_query)
    fact_lines = []
    review_rows = []
    total = 0

    for _, row in reranked_df.iterrows():
        rank = int(row["rank"])
        doc_id = str(row["doc_id"])
        doc = parse_rag_document(doc_id)
        snippet = select_relevant_snippet(doc, original_query, retrieval_query, intent)

        if not snippet:
            continue

        fact_line = build_fact_line(rank, doc, snippet)

        if total + len(fact_line) > max_total_chars and fact_lines:
            break

        fact_lines.append(fact_line)
        total += len(fact_line)

        review_rows.append({
            "rank": rank,
            "doc_id": doc_id,
            "rag_rerank_score": row.get("rag_rerank_score", ""),
            "rrf_score": row.get("rrf_score", row.get("score", "")),
            "bm25_rank": row.get("bm25_rank", ""),
            "dense_rank": row.get("dense_rank", ""),
            "title": doc.get("title", ""),
            "document_type": doc.get("document_type", ""),
            "course_code": doc.get("course_code", ""),
            "file_name": doc.get("file_name", ""),
            "snippet": snippet
        })

    return "\n".join(fact_lines), pd.DataFrame(review_rows)


# ----------------------------
# Prompting and answer quality guard
# ----------------------------
def build_contextualized_prompt(original_user_query, hidden_context):
    return f"""
Answer the student question using only the department facts below.
Use the facts as background context.
Do not mention document numbers, document IDs, file names, retrieval, chunks, BM25, FAISS, or RRF.
Give a direct advising-style answer.

If the facts do not contain the answer, say:
"The retrieved department documents do not provide enough information to answer that."

Student question:
{original_user_query}

Department facts:
{hidden_context}

Answer:
""".strip()


def build_conversation_continuation_prompt(current_user_query, previous_user_query, hidden_context):
    return f"""
Continue the advising conversation using the same department facts from the previous retrieval.
The current student message does not appear to introduce a new EECS retrieval topic, so treat it as a follow-up.

Do not mention document numbers, document IDs, file names, retrieval, chunks, BM25, FAISS, RRF, or hidden context.
Answer naturally and directly.

If the previous department facts are not enough to answer the follow-up, say:
"The previously retrieved department documents do not provide enough information to answer that."

Previous student question:
{previous_user_query}

Current follow-up:
{current_user_query}

Previously retrieved department facts:
{hidden_context}

Answer:
""".strip()


def clean_generated_answer(answer):
    answer = rag_normalize_text(answer)
    answer = re.sub(r"^(Answer:|Final answer:|Final answer for the user:)\s*", "", answer, flags=re.IGNORECASE).strip()
    return answer


def repeated_word_problem(answer):
    words = re.findall(r"[A-Za-z]+", str(answer).lower())
    if len(words) < 20:
        return False
    counts = {w: words.count(w) for w in set(words)}
    return max(counts.values()) / len(words) > 0.22


def answer_looks_bad(answer):
    if not answer or len(str(answer).strip()) < 12:
        return True

    lowered = str(answer).lower()
    bad_markers = [
        "doc id", "document 1", "document 2", "document 3", "document 4", "document 5", "document 6",
        "rrf", "bm25", "faiss", "pdf_", "course_", "retrieved department context",
        "file_name", "source_page", "chunk"
    ]

    if any(marker in lowered for marker in bad_markers):
        return True

    if repeated_word_problem(answer):
        return True

    if re.fullmatch(r"[A-Za-z0-9_\- .()]+\.pdf", str(answer).strip()):
        return True

    return False


def extract_course_suggestions_from_review(top_docs_df, max_items=6):
    suggestions = []
    seen = set()

    if top_docs_df is None or len(top_docs_df) == 0:
        return suggestions

    for _, row in top_docs_df.iterrows():
        code = str(row.get("course_code", "")).strip()
        title = str(row.get("title", "")).strip()
        key = code or title

        if not key or key.lower() in seen:
            continue

        seen.add(key.lower())

        if code and title:
            suggestions.append(f"{code} — {title}")
        elif title:
            suggestions.append(title)
        elif code:
            suggestions.append(code)

        if len(suggestions) >= max_items:
            break

    return suggestions


def extractive_fallback_answer(original_user_query, top_docs_df, continuation_mode=False):
    intent = classify_rag_intent(original_user_query)

    insufficient = (
        "The previously retrieved department documents do not provide enough information to answer that."
        if continuation_mode
        else "The retrieved department documents do not provide enough information to answer that."
    )

    if top_docs_df is None or len(top_docs_df) == 0:
        return insufficient

    snippets = [str(s).strip() for s in top_docs_df.get("snippet", []) if str(s).strip()]
    if not snippets:
        return insufficient

    if intent == "course_recommendation":
        suggestions = extract_course_suggestions_from_review(top_docs_df)
        if suggestions:
            return (
                "Based on the retrieved department course information, good options to review include "
                + "; ".join(suggestions[:6])
                + ". The student should still check the current schedule, prerequisites, and advisor guidance before registering."
            )

    if intent == "person_lookup":
        joined = " ".join(snippets[:4])
        m = re.search(
            r"(?:acting\s+chair|chair|director)\s*(?:is|:|-)?\s*([A-Z][A-Za-z.'-]+(?:\s+[A-Z][A-Za-z.'-]+){0,3})",
            joined
        )
        if m:
            return f"The retrieved department context indicates that {m.group(1)} is associated with that role."
        return "The retrieved department documents do not provide enough information to identify that person or role."

    strongest = []
    for s in snippets[:3]:
        s = re.sub(
            r"\b(Document_Type|Title|Course_Code|File_Name|Source_Page|Linked_File_or_Page)\s*=\s*",
            "",
            s
        )
        strongest.append(s)

    return "Based on the retrieved department context: " + " ".join(strongest)


def get_model_device(model_obj):
    try:
        return next(model_obj.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_chat_or_plain_prompt(prompt):
    if RAG_MODEL_KIND != "causal":
        return prompt

    system_message = (
        "You are a helpful academic advising assistant. "
        "Use only the supplied department context. "
        "Do not mention retrieval, chunks, filenames, document IDs, BM25, FAISS, or RRF. "
        "If the context does not answer the question, say that the retrieved department documents do not provide enough information."
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]

    if hasattr(rag_tokenizer, "apply_chat_template"):
        try:
            return rag_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return system_message + "\n\n" + prompt

    return system_message + "\n\n" + prompt


def decode_generated_tokens(outputs, inputs):
    if RAG_MODEL_KIND == "causal":
        input_len = inputs["input_ids"].shape[-1]
        generated = outputs[0][input_len:]
        return rag_tokenizer.decode(generated, skip_special_tokens=True)

    return rag_tokenizer.decode(outputs[0], skip_special_tokens=True)


def run_rag_model_once(prompt, max_input_tokens, max_new_tokens):
    formatted_prompt = build_chat_or_plain_prompt(prompt)

    inputs = rag_tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    )

    device = get_model_device(rag_model)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 1.15,
        "pad_token_id": rag_tokenizer.pad_token_id,
    }

    if RAG_MODEL_KIND == "seq2seq":
        generation_kwargs.update({
            "num_beams": 4,
            "early_stopping": True,
            "repetition_penalty": 1.25,
        })
    else:
        if rag_tokenizer.eos_token_id is not None:
            generation_kwargs["eos_token_id"] = rag_tokenizer.eos_token_id

    with torch.no_grad():
        outputs = rag_model.generate(**inputs, **generation_kwargs)

    return clean_generated_answer(decode_generated_tokens(outputs, inputs))


def generate_rag_answer(
    original_user_query,
    hidden_context,
    top_docs_df=None,
    max_input_tokens=RAG_MAX_INPUT_TOKENS,
    max_new_tokens=RAG_MAX_NEW_TOKENS,
    continuation_mode=False,
    previous_user_query=None
):
    if continuation_mode:
        prompt = build_conversation_continuation_prompt(
            current_user_query=original_user_query,
            previous_user_query=previous_user_query or "Previous question not available.",
            hidden_context=hidden_context
        )
    else:
        prompt = build_contextualized_prompt(
            original_user_query=original_user_query,
            hidden_context=hidden_context
        )

    answer = run_rag_model_once(prompt, max_input_tokens=max_input_tokens, max_new_tokens=max_new_tokens)

    if answer_looks_bad(answer):
        shorter_context = "\n".join(str(hidden_context).splitlines()[:5])

        if continuation_mode:
            retry_prompt = build_conversation_continuation_prompt(
                current_user_query=original_user_query,
                previous_user_query=previous_user_query or "Previous question not available.",
                hidden_context=shorter_context
            )
        else:
            retry_prompt = build_contextualized_prompt(
                original_user_query=original_user_query,
                hidden_context=shorter_context
            )

        retry_answer = run_rag_model_once(retry_prompt, max_input_tokens=1024, max_new_tokens=140)

        if not answer_looks_bad(retry_answer):
            return retry_answer

        return extractive_fallback_answer(original_user_query, top_docs_df, continuation_mode=continuation_mode)

    return answer


# ----------------------------
# Main RAG function and loop
# ----------------------------
LAST_RAG_RESULT = None


def ask_department_rag(
    original_user_query,
    top_k=RAG_TOP_K,
    retrieval_pool=RAG_RETRIEVAL_POOL,
    return_details=False,
    debug=RAG_DEBUG
):
    global LAST_RAG_RESULT

    original_user_query = str(original_user_query).strip()

    if not original_user_query:
        print("Please enter a non-empty query.")
        return None

    previous_result = LAST_RAG_RESULT if isinstance(LAST_RAG_RESULT, dict) else None

    has_previous_context = (
        previous_result is not None
        and str(previous_result.get("hidden_context", "")).strip()
        and previous_result.get("top_docs") is not None
    )

    should_refresh = should_refresh_retrieval_for_query(original_user_query)

    # First usable question always retrieves, even if no keyword is detected.
    if not has_previous_context:
        should_refresh = True

    if should_refresh:
        retrieval_query = improve_query_for_hybrid_search(original_user_query)

        fused_df = hybrid_search_custom_query(
            retrieval_query_text=retrieval_query,
            top_k=top_k,
            retrieval_pool=retrieval_pool
        )

        reranked_df = rerank_fused_results(
            fused_df=fused_df,
            original_query=original_user_query,
            retrieval_query=retrieval_query,
            top_k=top_k
        )

        hidden_context, top_docs_df = build_hidden_rag_context(
            reranked_df=reranked_df,
            original_query=original_user_query,
            retrieval_query=retrieval_query
        )

        continuation_mode = False
        previous_user_query = None

    else:
        # No major EECS keyword detected. Reuse the previous context.
        retrieval_query = previous_result.get("retrieval_query", "")
        hidden_context = previous_result.get("hidden_context", "")
        top_docs_df = previous_result.get("top_docs")
        fused_df = previous_result.get("fused_results")
        reranked_df = previous_result.get("reranked_results")

        continuation_mode = True
        previous_user_query = previous_result.get("original_query", "")

    if not str(hidden_context).strip():
        answer = (
            "The previously retrieved department documents do not provide enough information to answer that."
            if continuation_mode
            else "The retrieved department documents do not provide enough information to answer that."
        )
    else:
        answer = generate_rag_answer(
            original_user_query=original_user_query,
            hidden_context=hidden_context,
            top_docs_df=top_docs_df,
            continuation_mode=continuation_mode,
            previous_user_query=previous_user_query
        )

    LAST_RAG_RESULT = {
        "original_query": original_user_query,
        "retrieval_query": retrieval_query,
        "answer": answer,
        "top_docs": top_docs_df,
        "hidden_context": hidden_context,
        "fused_results": fused_df,
        "reranked_results": reranked_df,
        "intent": classify_rag_intent(original_user_query),
        "retrieval_refreshed": should_refresh,
        "continuation_mode": continuation_mode,
        "previous_user_query": previous_user_query,
        "has_major_eecs_keyword": query_has_major_eecs_keyword(original_user_query),
        "looks_like_followup": query_looks_like_followup(original_user_query)
    }

    print("\n" + "=" * 90)
    print("Contextualized RAG Answer")
    print("=" * 90)
    print(answer)

    if debug:
        print("\nDebug mode is on. The retrieved chunks are shown below for inspection only.")
        print("Original query:", original_user_query)
        print("Retrieval query:", retrieval_query)
        print("Retrieval refreshed:", should_refresh)
        print("Continuation mode:", continuation_mode)
        print("Major EECS keyword detected:", query_has_major_eecs_keyword(original_user_query))
        print("Looks like follow-up:", query_looks_like_followup(original_user_query))

        try:
            display(top_docs_df)
        except Exception:
            print(top_docs_df)

    if return_details:
        return LAST_RAG_RESULT

    return answer


def rag_query_loop():
    print("Interactive CS/EE Department RAG Assistant")
    print("Enter a custom question.")
    print("Fresh hybrid search runs when the query contains major EECS keywords.")
    print("Follow-up questions without major EECS keywords reuse the previous retrieved context.")
    print("Only the final contextualized answer is displayed.")
    print("Type 'exit', 'quit', or 'q' to stop.")

    while True:
        user_query = input("\nEnter your CS/EE department query: ").strip()

        if user_query.lower() in {"exit", "quit", "q"}:
            print("Stopping interactive RAG loop.")
            break

        if not user_query:
            print("Please enter a question or type 'exit' to stop.")
            continue

        _ = ask_department_rag(user_query, return_details=False, debug=False)


# Start the interactive loop.
rag_query_loop()


Loading Hugging Face RAG answer model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

RAG answer model loaded as: causal
Interactive CS/EE Department RAG Assistant
Enter a custom question.
Fresh hybrid search runs when the query contains major EECS keywords.
Follow-up questions without major EECS keywords reuse the previous retrieved context.
Only the final contextualized answer is displayed.
Type 'exit', 'quit', or 'q' to stop.



Enter your CS/EE department query:  How many credits are required for the artificial intelligence MS?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...


Contextualized RAG Answer
The requirements for an MS degree in Computer Science typically include: 1. Completing 15-18 credits of coursework beyond your bachelor's degree. 2. Submitting a program worksheet detailing your chosen electives and any additional courses you plan to take. 3. Adhering to specific elective choices within the EECS department based on your interests and career goals. 4. Participating in seminars and possibly writing a thesis or completing a project under faculty supervision. For more detailed guidance tailored specifically to the Computer Engineering MS program, please refer to Fact 2 provided above. This fact outlines the core components of the program including the number of credits needed, submission of materials like the program worksheet, and the structure of the curriculum.



Enter your CS/EE department query:  What is required if an AI MS graduate students wants a big data analytics certificate?



Contextualized RAG Answer
To obtain a big-data analytics certificate after graduating with an AI Master's degree, your student should focus on selecting at least one core course related to big data analysis within the specified tracks offered by the university. Specifically: 1. **Core Courses**: - From the "Applications Track" (if applicable), select at least two courses such as `CAP 5545` (Big Data Analytics Fundamentals) or similar. 2. **Electives**: - Additionally, take elective courses like `CAP6765 Advanced Data Management Systems`, which could complement the big data requirements. The key points here are ensuring compliance with the specific curriculum guidelines provided by the institution regarding both core and elective courses necessary for obtaining the desired certification.



Enter your CS/EE department query:  quit


Stopping interactive RAG loop.
